# 1. Yelp — tải dữ liệu và khám phá

## Mục tiêu

Tải [Yelp-JSON.zip từ Google Drive](https://drive.google.com/file/d/1cyir0oGMviwUjXPtMhvxpX27TQL2Lg6y/view?usp=drive_link), xác minh file, chuẩn bị bảng business/check-in và khám phá phạm vi dữ liệu. Archive khoảng **4,35 GB**; chỉ lấy một mẫu review giới hạn.

Check-in là **hoạt động được ghi nhận**, không phải số khách duy nhất, doanh thu hay lượt bán. Mẫu review lấy từ đầu file nên không đại diện cho toàn bộ reviews. Notebook này chỉ tải dữ liệu và EDA; không huấn luyện mô hình, ghép weather hay kết luận quan hệ nhân quả.

Chạy từ trên xuống dưới. ZIP nằm trong `data/downloads/yelp.zip`, bảng nguồn trong `data/raw/yelp/`, kết quả khám phá trong `data/processed/yelp/`. Các file dữ liệu và bản notebook đã chạy không đưa vào Git.

## Chuẩn bị

Chạy từ thư mục gốc repository:

```sh
uv sync --frozen --group notebooks
uv run --frozen --group notebooks jupyter lab --notebook-dir=.
```

Trong VS Code, chọn kernel `.venv/bin/python`. Trên Colab, tải notebook lên và chạy `%pip install gdown pandas matplotlib` trong một cell riêng. Dữ liệu local của Colab mất khi runtime kết thúc; dùng `SITESENSE_YELP_ARCHIVE` để trỏ đến bản ZIP đã lưu nếu cần.

Cần khoảng **5 GB trống** cho ZIP và các bảng được stage. Notebook hỗ trợ JSON trực tiếp trong ZIP hoặc một TAR nén bên trong ZIP; không giải nén toàn bộ reviews/users/tips. Các lần chạy sau dùng lại file nguồn đã xác minh.

In [ ]:
import hashlib
import json
import os
import platform
import shutil
import tarfile
import zipfile
from datetime import UTC, datetime
from importlib.metadata import version
from itertools import islice
from pathlib import Path

import gdown
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 14)
pd.set_option("display.max_colwidth", 60)
plt.rcParams.update({"figure.figsize": (11, 4), "axes.grid": True, "grid.alpha": 0.25})
print({package: version(package) for package in ("gdown", "pandas", "matplotlib")})

### Tham số

`ARCHIVE_PATH` có thể trỏ đến ZIP tải thủ công. Chỉ đặt `EXPECTED_ARCHIVE_BYTES = None` khi chủ động dùng một phiên bản khác. `BUSINESS_ID = None` chọn business có nhiều check-in nhất trong thành phố/ngành đã chọn để xem ví dụ mô tả.

Business và check-in được đọc đầy đủ. Review giới hạn ở `REVIEW_SAMPLE_ROWS`; bảng user và tip chỉ được liệt kê trong archive.

In [ ]:
PROJECT_ROOT = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "pyproject.toml").is_file()
        and (path / "notebooks/01_yelp_dataset_exploration.ipynb").is_file()
    ),
    Path.cwd(),
)
DOWNLOAD_DIR = PROJECT_ROOT / "data/downloads"
STAGING_DIR = PROJECT_ROOT / "data/raw/yelp"
OUTPUT_DIR = PROJECT_ROOT / "data/processed/yelp"
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DRIVE_FILE_ID = "1cyir0oGMviwUjXPtMhvxpX27TQL2Lg6y"
SOURCE_URL = f"https://drive.google.com/file/d/{DRIVE_FILE_ID}/view?usp=drive_link"
ARCHIVE_PATH = Path(os.environ.get("SITESENSE_YELP_ARCHIVE", DOWNLOAD_DIR / "yelp.zip"))
# Size observed in Drive metadata on 2026-10-05; change for a different archive version.
EXPECTED_ARCHIVE_BYTES = None if os.environ.get("SITESENSE_YELP_ARCHIVE") else 4_345_335_132
COMPUTE_SHA256 = True

CITY = "Philadelphia"
STATE = "PA"
CATEGORY = "Restaurants"  # Exact Yelp category, or None to include all categories.
BUSINESS_ID = None
REVIEW_SAMPLE_ROWS = 10_000

assert REVIEW_SAMPLE_ROWS > 0
print(f"Archive: {ARCHIVE_PATH}")
print(f"Outputs: {OUTPUT_DIR}")

## Các bước

### 1. Tải và xác minh archive

`gdown` hỗ trợ xác nhận tải file lớn và tiếp tục file `.part` khi bị ngắt. Nếu Drive chặn tải ẩn danh, mở link nguồn, tải thủ công vào `ARCHIVE_PATH`, rồi chạy lại. Notebook không lưu thông tin đăng nhập Drive.

Kiểm tra dung lượng và định dạng ZIP phát hiện file tải thiếu/sai. SHA-256 ghi nhận chính xác bản local đã dùng; chưa có digest do nguồn phát hành để xác thực tính chính chủ.

In [ ]:
if not ARCHIVE_PATH.exists():
    ARCHIVE_PATH.parent.mkdir(parents=True, exist_ok=True)
    required_space = (EXPECTED_ARCHIVE_BYTES or 4_345_335_132) + 512 * 1024**2
    partial_bytes = sum(
        path.stat().st_size for path in ARCHIVE_PATH.parent.glob(f"{ARCHIVE_PATH.name}*.part")
    )
    if shutil.disk_usage(ARCHIVE_PATH.parent).free < max(0, required_space - partial_bytes):
        raise OSError("Not enough disk space for the ZIP. Free space or change ARCHIVE_PATH.")
    downloaded = gdown.download(
        id=DRIVE_FILE_ID, output=str(ARCHIVE_PATH), resume=True, use_cookies=False
    )
    if downloaded is None:
        raise RuntimeError(f"Download failed. Download manually from {SOURCE_URL}.")
else:
    print("Reusing the local archive.")

archive_bytes = ARCHIVE_PATH.stat().st_size
if EXPECTED_ARCHIVE_BYTES is not None and archive_bytes != EXPECTED_ARCHIVE_BYTES:
    raise ValueError(
        f"ZIP size mismatch: {archive_bytes:,} vs expected {EXPECTED_ARCHIVE_BYTES:,} bytes. "
        "Check the download; use EXPECTED_ARCHIVE_BYTES=None only for a known different version."
    )
if not zipfile.is_zipfile(ARCHIVE_PATH):
    raise ValueError("The downloaded file is not a ZIP. Check for a Drive HTML error page.")


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024**2), b""):
            digest.update(chunk)
    return digest.hexdigest()


archive_sha256 = sha256_file(ARCHIVE_PATH) if COMPUTE_SHA256 else None
print(f"ZIP size: {archive_bytes / 1e9:.2f} GB; SHA-256: {archive_sha256 or 'skipped'}")

### 2. Xem cấu trúc và stage các bảng cần thiết

ZIP được chia sẻ chứa một `yelp_dataset.tar` nén gzip. Đọc tuần tự một lần để lưu đầy đủ business/check-in và tối đa `REVIEW_SAMPLE_ROWS` reviews. Output dùng tên cố định, không lấy đường dẫn từ archive và không giải nén symbolic links.

Manifest lưu thông tin archive, dung lượng và SHA-256 của các bảng staged để kiểm tra khi dùng lại. Manifest cũ chưa có checksum được bổ sung ở lần chạy đầu; các bước dưới vẫn đọc và kiểm tra JSON của toàn bộ business/check-in. Mẫu review không kiểm tra toàn bộ bảng review.

In [ ]:
with zipfile.ZipFile(ARCHIVE_PATH) as archive:
    inventory = pd.DataFrame(
        [
            {
                "member": member.filename,
                "compressed_mb": member.compress_size / 1e6,
                "uncompressed_mb": member.file_size / 1e6,
            }
            for member in archive.infolist()
            if not member.is_dir()
        ]
    )
display(inventory.head(20).round(2))
print(
    "Outer ZIP payload (before expanding nested TAR): "
    f"{inventory.uncompressed_mb.sum() / 1000:.2f} GB"
)

In [ ]:
tables = ("business", "checkin", "review")
staged_paths = {table: STAGING_DIR / f"{table}.jsonl" for table in tables}
STAGING_DIR.mkdir(parents=True, exist_ok=True)
staging_manifest_path = STAGING_DIR / "staging_manifest.json"
staging_key = {
    "archive_path": str(ARCHIVE_PATH.resolve()),
    "archive_sha256": archive_sha256,
    "archive_bytes": archive_bytes,
    "archive_mtime_ns": ARCHIVE_PATH.stat().st_mtime_ns,
    "review_sample_rows": REVIEW_SAMPLE_ROWS,
    "staging_version": 1,
}
cached = None
if staging_manifest_path.is_file():
    try:
        candidate = json.loads(staging_manifest_path.read_text())
        if (
            isinstance(candidate, dict)
            and candidate.get("key") == staging_key
            and all(
                path.is_file()
                and path.stat().st_size == candidate.get("staged_bytes", {}).get(table)
                for table, path in staged_paths.items()
            )
        ):
            fingerprints = {table: sha256_file(path) for table, path in staged_paths.items()}
            if not candidate.get("staged_sha256") or fingerprints == candidate["staged_sha256"]:
                cached = candidate
    except json.JSONDecodeError:
        pass  # An interrupted/invalid staging manifest is rebuilt below.


def table_for_member(name):
    if name.startswith("__MACOSX/"):
        return None
    for table in tables:
        if Path(name).name.lower() in {
            f"yelp_academic_dataset_{table}.json",
            f"{table}.json",
            f"{table}.jsonl",
        }:
            return table
    return None


members = {}
inner_inventory = []


def stage_table(table, member_name, binary):
    if table in members:
        raise ValueError(f"Duplicate {table} members; inspect archive before choosing a source.")
    temporary_path = staged_paths[table].with_suffix(".partial")
    with temporary_path.open("wb") as output:
        if table == "review":
            nonempty_lines = (line for line in binary if line.strip())
            for line in islice(nonempty_lines, REVIEW_SAMPLE_ROWS):
                output.write(line)
        else:
            shutil.copyfileobj(binary, output, length=8 * 1024**2)
    temporary_path.replace(staged_paths[table])
    members[table] = member_name
    print(f"Staged {table}: {staged_paths[table].stat().st_size / 1e6:.1f} MB", flush=True)

In [ ]:
if cached:
    members = cached["members"]
    inner_inventory = cached["inner_inventory"]
    print("Reusing staged JSON tables.")
else:
    with zipfile.ZipFile(ARCHIVE_PATH) as archive:
        direct_members = [
            member
            for member in archive.infolist()
            if not member.is_dir() and table_for_member(member.filename)
        ]
        if direct_members:
            for member in direct_members:
                with archive.open(member) as binary:
                    stage_table(table_for_member(member.filename), member.filename, binary)
        else:
            nested_members = [
                member
                for member in archive.infolist()
                if not member.filename.startswith("__MACOSX/")
                and member.filename.lower().endswith((".tar", ".tar.gz", ".tgz"))
            ]
            if len(nested_members) != 1:
                raise ValueError(
                    "Expected JSON tables or one nested TAR. Inspect the ZIP inventory."
                )
            with archive.open(nested_members[0]) as binary:
                with tarfile.open(fileobj=binary, mode="r|*") as inner:
                    for member in inner:
                        if not member.isfile():
                            continue
                        inner_inventory.append(
                            {"member": member.name, "uncompressed_mb": member.size / 1e6}
                        )
                        table = table_for_member(member.name)
                        if table:
                            with inner.extractfile(member) as source:
                                stage_table(table, member.name, source)
                # Consume remaining ZIP bytes, including any trailer, for its CRC check.
                for _ in iter(lambda: binary.read(8 * 1024**2), b""):
                    pass
    if set(members) != set(tables):
        raise ValueError(f"Missing required tables: {set(tables) - set(members)}")
    cached = {
        "key": staging_key,
        "members": members,
        "inner_inventory": inner_inventory,
        "staged_bytes": {table: path.stat().st_size for table, path in staged_paths.items()},
    }

staged_sha256 = {table: sha256_file(path) for table, path in staged_paths.items()}
cached["staged_sha256"] = staged_sha256
manifest_partial = staging_manifest_path.with_suffix(".partial")
manifest_partial.write_text(json.dumps(cached, indent=2) + "\n")
manifest_partial.replace(staging_manifest_path)

if inner_inventory:
    display(pd.DataFrame(inner_inventory).head(20).round(2))
    inner_payload_gb = sum(row["uncompressed_mb"] for row in inner_inventory) / 1000
    print(f"Nested TAR files: {len(inner_inventory):,}; total payload: {inner_payload_gb:.2f} GB")
display(pd.Series(members, name="Source member").to_frame())


def records(table, limit=None):
    with staged_paths[table].open(encoding="utf-8-sig") as stream:
        nonempty_lines = (line for line in stream if line.strip())
        lines = nonempty_lines if limit is None else islice(nonempty_lines, limit)
        for row_number, line in enumerate(lines, start=1):
            try:
                record = json.loads(line)
            except json.JSONDecodeError as error:
                raise ValueError(f"Invalid JSON in {table}, record {row_number}") from error
            if not isinstance(record, dict):
                raise ValueError(f"Expected a JSON object in {table}, record {row_number}")
            yield record


display({table: list(next(records(table)).keys()) for table in members})

### 3. Đọc business và kiểm tra chất lượng

Chỉ đưa các trường phẳng và categories vào DataFrame; `attributes`/`hours` lồng nhau vẫn nằm trong nguồn. ID thiếu/trùng, tọa độ sai và categories thiếu ảnh hưởng đến thống kê và phép ghép.

In [ ]:
business_fields = [
    "business_id",
    "name",
    "city",
    "state",
    "latitude",
    "longitude",
    "stars",
    "review_count",
    "is_open",
    "categories",
]
business = pd.DataFrame(
    ({field: record.get(field) for field in business_fields} for record in records("business")),
    columns=business_fields,
)
if business.empty or business.business_id.isna().any() or business.business_id.duplicated().any():
    raise ValueError("Business IDs must be present and unique for a reliable join.")
for field in ("latitude", "longitude", "stars", "review_count", "is_open"):
    business[field] = pd.to_numeric(business[field], errors="coerce")

quality = pd.DataFrame(
    {
        "dtype": business.dtypes.astype(str),
        "missing_rows": business.isna().sum(),
        "missing_percent": business.isna().mean().mul(100).round(2),
    }
)
display(quality)
display(business.head(5))
invalid_coordinates = ~(business.latitude.between(-90, 90) & business.longitude.between(-180, 180))
print(
    f"Businesses: {len(business):,}; invalid/missing coordinate rows: {invalid_coordinates.sum():,}"
)
print(f"Missing category rows: {business.categories.isna().sum():,}")

### 4. Khám phá địa lý và ngành

Phạm vi dataset phản ánh quá trình thu thập. Nhiều business/check-in không tự chứng minh một thị trường phù hợp để mở cửa hàng. Một business có thể có nhiều categories nên các nhóm ngành chồng lấp.

In [ ]:
business["category_list"] = business.categories.fillna("").map(
    lambda value: [category.strip() for category in value.split(",") if category.strip()]
)
city_coverage = (
    business.groupby(["state", "city"], dropna=False)
    .agg(
        businesses=("business_id", "size"),
        median_stars=("stars", "median"),
        listed_open_share=("is_open", "mean"),
    )
    .sort_values("businesses", ascending=False)
)
display(city_coverage.head(15).round(3))
top_categories = business.category_list.explode().dropna().value_counts().head(15)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
city_coverage.head(12).businesses.sort_values().plot.barh(ax=axes[0])
axes[0].set(title="Cities with most listed businesses", xlabel="Businesses", ylabel="State, city")
top_categories.sort_values().plot.barh(ax=axes[1])
axes[1].set(
    title="Common business categories (overlapping)", xlabel="Businesses", ylabel="Category"
)
plt.tight_layout()
plt.show()

### 5. Tóm tắt check-in

Đọc một record cho mỗi business; đếm timestamp mà không mở rộng toàn bộ thành một bảng sự kiện lớn. Không có record check-in khác với có record nhưng không có timestamp.

Khoảng thời gian toàn dataset được suy ra từ timestamp đầu/cuối của từng record, với định dạng nguồn cố định. Đây là kiểm tra hai đầu khoảng thời gian, không kiểm tra cú pháp của từng sự kiện. Bước 7 parse toàn bộ timestamp cho business minh họa.

In [ ]:
checkin_rows = []
for record in records("checkin"):
    dates = record.get("date") or ""
    tokens = [token.strip() for token in dates.split(",") if token.strip()]
    checkin_rows.append(
        {
            "business_id": record.get("business_id"),
            "timestamp_entries": len(tokens),
            "first_timestamp": min(tokens, default=None),
            "last_timestamp": max(tokens, default=None),
        }
    )
checkin = pd.DataFrame(
    checkin_rows, columns=["business_id", "timestamp_entries", "first_timestamp", "last_timestamp"]
)
if checkin.business_id.isna().any() or checkin.business_id.duplicated().any():
    raise ValueError("Check-in IDs must be present and unique; inspect source before joining.")

for field in ("first_timestamp", "last_timestamp"):
    checkin[field] = pd.to_datetime(checkin[field], format="%Y-%m-%d %H:%M:%S", errors="coerce")
invalid_endpoints = (
    checkin.loc[checkin.timestamp_entries.gt(0), ["first_timestamp", "last_timestamp"]]
    .isna()
    .sum()
    .sum()
)
if invalid_endpoints:
    raise ValueError(
        f"Unparseable check-in endpoints: {invalid_endpoints:,}; inspect source dates."
    )
print(
    f"Check-in records: {len(checkin):,}; timestamp entries: {checkin.timestamp_entries.sum():,}; "
    f"span: {checkin.first_timestamp.min()} to {checkin.last_timestamp.max()}"
)

unmatched_checkin = ~checkin.business_id.isin(business.business_id)
print(f"Check-in rows without business metadata: {unmatched_checkin.sum():,}")
business_activity = business.merge(checkin, how="left", on="business_id", validate="one_to_one")
business_activity["has_checkin_record"] = business_activity.timestamp_entries.notna()
activity_coverage = business_activity.groupby(["state", "city"], dropna=False).agg(
    businesses=("business_id", "size"),
    businesses_with_checkin_records=("has_checkin_record", "sum"),
    timestamp_entries=("timestamp_entries", "sum"),
)
activity_coverage["checkin_record_share"] = (
    activity_coverage.businesses_with_checkin_records / activity_coverage.businesses
)
display(activity_coverage.sort_values("timestamp_entries", ascending=False).head(15).round(3))
display(business_activity.timestamp_entries.describe(percentiles=[0.5, 0.9, 0.99]).to_frame())

### 6. Xem mẫu review giới hạn

Chỉ lấy `REVIEW_SAMPLE_ROWS` reviews đầu file để xem schema và missingness. Phân phối sao của mẫu này không ước lượng cho toàn bộ Yelp. Không hiển thị nội dung review hay danh tính người viết; chỉ giữ độ dài text.

In [ ]:
reviews = pd.DataFrame(
    [
        {
            "review_id": record.get("review_id"),
            "business_id": record.get("business_id"),
            "stars": record.get("stars"),
            "date": record.get("date"),
            "text_characters": len(record.get("text") or ""),
            "useful": record.get("useful"),
        }
        for record in records("review", limit=REVIEW_SAMPLE_ROWS)
    ]
)
if reviews.empty:
    raise ValueError("The review file contains no records.")
reviews["date"] = pd.to_datetime(reviews.date, errors="coerce")
reviews["stars"] = pd.to_numeric(reviews.stars, errors="coerce")
display(reviews.drop(columns=["review_id"]).head(5))
display(reviews[["stars", "text_characters", "useful"]].describe().round(2))
print(
    f"Prefix sample: {len(reviews):,} rows; duplicate IDs: {reviews.review_id.duplicated().sum():,}"
)
print(f"Unparseable review dates: {reviews.date.isna().sum():,}")
print(f"Sample date range: {reviews.date.min()} to {reviews.date.max()}")
reviews.stars.value_counts().sort_index().plot.bar(
    title=f"Ratings in the first {len(reviews):,} reviews (prefix sample)",
    xlabel="Stars",
    ylabel="Sampled reviews",
    rot=0,
)
plt.tight_layout()
plt.show()

### 7. Mô tả chuỗi hoạt động của một business

Mặc định chọn business nhiều check-in nhất trong thành phố/ngành: ví dụ này không đại diện cho mọi business. Timestamp không có timezone offset; giữ nguyên nguồn và chưa coi là ngày local để ghép weather.

Chỉ điền 0 **giữa sự kiện đầu tiên và cuối cùng**. Không có check-in ghi nhận không có nghĩa không có khách hay business đóng cửa. Giữ timestamp trùng vì nguồn không cho biết chúng có phải các check-in khác nhau hay không.

In [ ]:
eligible = business_activity[business_activity.timestamp_entries.fillna(0).gt(0)].copy()
if BUSINESS_ID is None:
    eligible = eligible[eligible.city.str.casefold().eq(CITY.casefold())]
    if STATE:
        eligible = eligible[eligible.state.str.casefold().eq(STATE.casefold())]
    if CATEGORY:
        eligible = eligible[eligible.category_list.map(lambda categories: CATEGORY in categories)]
else:
    eligible = eligible[eligible.business_id.eq(BUSINESS_ID)]
if eligible.empty:
    raise ValueError(
        "No eligible business. Change CITY, STATE, CATEGORY, or BUSINESS_ID using coverage tables."
    )

selected = eligible.sort_values(["timestamp_entries", "business_id"], ascending=[False, True]).iloc[
    0
]
selected_id = selected.business_id
display(selected.drop(labels=["category_list"]).to_frame("Selected business"))
selected_record = next(
    record for record in records("checkin") if record["business_id"] == selected_id
)
date_tokens = [token.strip() for token in selected_record["date"].split(",") if token.strip()]
timestamps = pd.Series(pd.to_datetime(date_tokens, format="%Y-%m-%d %H:%M:%S", errors="coerce"))
invalid_timestamps = int(timestamps.isna().sum())
repeated_timestamps = int(timestamps.dropna().duplicated().sum())
print(
    f"Unparseable timestamps: {invalid_timestamps:,}; "
    f"repeated timestamps retained: {repeated_timestamps:,}"
)
if timestamps.notna().sum() == 0:
    raise ValueError("The selected business has no parseable check-in timestamps.")

day_counts = timestamps.dropna().dt.normalize().value_counts().sort_index()
observed_days = pd.date_range(day_counts.index.min(), day_counts.index.max(), freq="D")
daily = day_counts.reindex(observed_days, fill_value=0).rename("recorded_checkins").to_frame()
daily.index.name = "date"
daily["weekday"] = daily.index.dayofweek
assert int(daily.recorded_checkins.sum()) == int(timestamps.notna().sum())
assert len(date_tokens) == int(selected.timestamp_entries)
display(daily.head(7))
print(
    f"Observed event span: {daily.index.min().date()} to {daily.index.max().date()}; "
    f"{len(daily):,} days"
)
print(f"Days with zero recorded check-ins: {daily.recorded_checkins.eq(0).mean():.1%}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7))
daily.recorded_checkins.resample("MS").sum().plot(ax=axes[0])
axes[0].set(
    title=f"Monthly recorded check-ins — {selected['name']}",
    xlabel="Month",
    ylabel="Recorded check-ins",
)
weekday_means = daily.groupby("weekday").recorded_checkins.mean().reindex(range(7))
weekday_means.index = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
weekday_means.plot.bar(ax=axes[1], rot=0)
axes[1].set(
    title="Weekday pattern across the observed span (descriptive)",
    xlabel="Weekday",
    ylabel="Mean daily recorded check-ins",
)
plt.tight_layout()
plt.show()

## Kiểm tra

Đối chiếu số dòng sau phép ghép, tổng timestamp và chuỗi theo ngày; lưu phiên bản package, fingerprint nguồn, tham số và missingness cùng các bảng nhỏ. Kết quả xác minh luồng EDA, không chứng minh tính đại diện của dataset.

Rerun thay thế các output hiện tại. Các file forecast do phiên bản cũ của notebook tạo ra được xóa riêng theo tên; không xóa dữ liệu nguồn hay kết quả nghiên cứu khác.

In [ ]:
assert len(business_activity) == len(business)
matched_entries = business_activity.timestamp_entries.sum()
source_matched_entries = checkin.loc[~unmatched_checkin, "timestamp_entries"].sum()
assert matched_entries == source_matched_entries
assert int(daily.recorded_checkins.sum()) + invalid_timestamps == len(date_tokens)
assert len(reviews) <= REVIEW_SAMPLE_ROWS
assert all(sha256_file(path) == staged_sha256[table] for table, path in staged_paths.items())
print(
    "Passed: unique IDs, join row count, timestamp totals, review bound, unchanged staged inputs."
)

city_coverage.to_csv(OUTPUT_DIR / "business_city_coverage.csv")
activity_coverage.to_csv(OUTPUT_DIR / "checkin_city_coverage.csv")
daily.to_csv(OUTPUT_DIR / "selected_business_daily_checkins.csv")
# Retire only outputs owned by the removed forecasting section.
for filename in ("baseline_metrics.csv", "holdout_predictions.csv", "experiment_manifest.json"):
    (OUTPUT_DIR / filename).unlink(missing_ok=True)

manifest = {
    "status": "complete",
    "run_at_utc": datetime.now(UTC).isoformat(),
    "source_url": SOURCE_URL,
    "archive_name": ARCHIVE_PATH.name,
    "archive_bytes": archive_bytes,
    "archive_sha256": archive_sha256,
    "staged_sha256": staged_sha256,
    "python": platform.python_version(),
    "packages": {package: version(package) for package in ("gdown", "pandas", "matplotlib")},
    "members": members,
    "parameters": {
        "city": CITY,
        "state": STATE,
        "category": CATEGORY,
        "requested_business_id": BUSINESS_ID,
        "review_sample_rows": REVIEW_SAMPLE_ROWS,
    },
    "business_rows": len(business),
    "businesses_missing_categories": int(business.categories.isna().sum()),
    "businesses_invalid_coordinates": int(invalid_coordinates.sum()),
    "checkin_rows": len(checkin),
    "checkin_timestamp_entries": int(checkin.timestamp_entries.sum()),
    "checkin_start": checkin.first_timestamp.min().isoformat(),
    "checkin_end": checkin.last_timestamp.max().isoformat(),
    "businesses_without_checkin_record": int((~business_activity.has_checkin_record).sum()),
    "unmatched_checkin_rows": int(unmatched_checkin.sum()),
    "review_rows_sampled": len(reviews),
    "review_invalid_dates": int(reviews.date.isna().sum()),
    "review_duplicate_ids": int(reviews.review_id.duplicated().sum()),
    "selected_business_id": selected_id,
    "selected_business_name": selected["name"],
    "invalid_selected_timestamps": invalid_timestamps,
    "repeated_selected_timestamps_retained": repeated_timestamps,
    "selected_timestamp_entries": len(date_tokens),
    "selected_event_days": len(daily),
    "limitations": [
        "checkins are recorded activity, not customers or revenue",
        "review sample is a file prefix",
        "illustrative business selected using its full activity history",
        "source timestamps have no timezone offset",
        "dataset time coverage checks record endpoints, not every timestamp",
        "no weather join or causal estimate",
    ],
}
(OUTPUT_DIR / "exploration_manifest.json").write_text(json.dumps(manifest, indent=2) + "\n")
summary = pd.Series(
    {
        "Business rows": manifest["business_rows"],
        "Check-in records": manifest["checkin_rows"],
        "Recorded timestamp entries": manifest["checkin_timestamp_entries"],
        "Business without check-in record": manifest["businesses_without_checkin_record"],
        "Check-in source start": manifest["checkin_start"],
        "Check-in source end": manifest["checkin_end"],
        "Reviews in prefix sample": manifest["review_rows_sampled"],
    },
    name="Observed value",
)
display(summary.to_frame())
display(
    pd.DataFrame({"output": sorted(path.name for path in OUTPUT_DIR.iterdir() if path.is_file())})
)

## Kết luận

Sau khi chạy, bảng tóm tắt ở trên cho biết số business, độ phủ check-in, khoảng thời gian nguồn và mẫu review thực tế. Sửa `CITY`, `STATE`, `CATEGORY` hoặc `BUSINESS_ID` rồi chạy lại để đổi ví dụ EDA.

Check-in vẫn chỉ là tín hiệu hoạt động; mẫu review là prefix và chuỗi minh họa ưu tiên business nhiều hoạt động. Cần làm rõ timezone, độ phủ và ngày thiếu trước khi kết hợp weather trong notebook Climate. Không đưa forecast hay kết luận nhân quả vào bước khám phá dữ liệu này.

Xem [nguyên tắc nghiên cứu](../docs/agents/data-interpretation.md) và [điều khoản Yelp Open Dataset](https://www.yelp.com/dataset) trước khi chia sẻ lại dữ liệu nguồn.